# Pilot 2025 global model selection and DOE

This notebook documents a full deterministic workflow for the pilot-scale fermentation data:

1. Load the curated pilot data and the previous ethyl-acetate aroma model.
2. Diagnose state-level model adequacy before re-fitting any new structure.
3. Compare secondary-metabolite ODE variants for pyruvate, acetaldehyde, and acetate.
4. Select a final ODE structure using weighted residual metrics, information criteria, and boundary diagnostics.
5. Build the current-data Fisher Information Matrix (FIM) with Pyomo DoE-compatible scaling.
6. Rank natural-must candidate experiments using model-based DOE metrics.

The distinction used here is:

$$r(\theta)=\frac{\hat{y}(\theta)-y}{\sigma_y}$$

$$J=\frac{\partial r}{\partial \log\theta}, \qquad F=J^\top J.$$

Poor estimability means that $F$ has weak directions after a model can already reproduce the data. Poor structural adequacy means that the residuals remain biased or shape-incompatible before the FIM interpretation is meaningful.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

RESULTS = Path('results/global_state_model_selection_doe')
print(RESULTS.resolve())


## Data and curation

The pilot workbook is treated as natural-must data. Online CO2 files are curated as in the previous pilot workflow: unusable CO2 files for batches 25150 and 25151 are excluded, and batch 25171 is shifted so that the sustained CO2 activation point is the effective fermentation start.

Dissolved oxygen is not used as a pilot observation here because the current pilot workbook does not expose a usable DO column; O2 remains a latent state in the secondary model.


In [ ]:
pd.read_csv(RESULTS / 'co2_curation_decisions.csv')

## Observation coverage

The adequacy diagnostic is only interpreted for states with actual observations. States without observations can still affect the FIM through model coupling and future DOE outputs, but their current-data residual fit is not directly testable.


In [ ]:
pd.read_csv(RESULTS / 'observation_coverage.csv')

## State-level adequacy diagnostic

For each observed state, species, and pool, the diagnostic computes:

$$RMSE=\sqrt{\frac{1}{n}\sum_i(\hat{y}_i-y_i)^2}$$

$$rRMSE=\frac{RMSE}{\max(\operatorname{median}(|y|),0.25(\max y-\min y),\epsilon)}$$

$$rBias=\frac{\operatorname{mean}(\hat{y}-y)}{\max(\operatorname{median}(|y|),0.25(\max y-\min y),\epsilon)}.$$

The flag is structural when normalized error is high and/or residuals show systematic shape incompatibility. Limited data are not interpreted as structural failure.


In [ ]:
initial = pd.read_csv(RESULTS / 'adequacy_initial_current.csv')
cols = [c for c in ['group','state','species','pool','n','relative_rmse','relative_bias','corr','adequacy_flag','adequacy_reason'] if c in initial.columns]
initial.sort_values(['adequacy_flag','relative_rmse'], ascending=[False, False])[cols]


## Secondary-state structural variants

The selected aroma model from the previous workflow is kept fixed while secondary-metabolite structure is tested. This avoids confusing ethyl-acetate structural error with pyruvate, acetaldehyde, or acetate structural error.

The common gating terms are:

$$\phi_N=\frac{N}{N+K_N}, \qquad \phi_{stat}=1-\phi_N,$$

$$g_{O2}=\frac{O_2}{O_2+K_{O2}}, \qquad \phi_{ana}=\frac{K_{O2}}{O_2+K_{O2}}, \qquad \phi_E=\frac{E}{E+K_E}.$$

Candidate structures are compared using robust log-parameter least squares. The model-selection score is:

$$Score = BIC + P_{bounds}+P_{adequacy}.$$

This penalizes models that fit by driving parameters to bounds or by keeping large state-level residual bias.


In [ ]:
pd.read_csv(RESULTS / 'secondary_model_selection_summary.csv')

## Selected secondary structure

Selected structure: `secondary_full_chem_o2fixed`

Full chemical secondary layer with O2 transfer parameters fixed. This is the upper-complexity check: it is accepted only if its improved residuals justify the extra degrees of freedom.

$$\frac{dPyr}{dt}=(k_{PyrS,N}\phi_N+k_{PyrS,stat}\phi_{stat})q_S+k_{PyrO2}g_{O2}X-k_{PyrDrain}Pyr X(0.25+\phi_{stat}+0.5\phi_E)$$

$$\frac{dAcAld}{dt}=k_{AldPyr}PyrX+(k_{AldS,N}\phi_N+k_{AldS,stat}\phi_{stat})q_S+k_{AldO2}g_{O2}X-k_{AldRed}AcAldX(\phi_{ana}+0.25\phi_{stat})-k_{AcAld}AcAldXg_{O2}$$

$$\frac{dAcetate}{dt}=\frac{k_{AcAld}AcAldXg_{O2}}{1000}+k_{AcStress}X\phi_E-k_{AcAssim}AcetateX\phi_N$$


In [ ]:
pd.read_csv(RESULTS / 'theta_selected_global_model.csv', index_col=0).head(60)

## Final adequacy after structural selection

The table below repeats the adequacy diagnostic after substituting the selected secondary structure into the global model. States that remain flagged are not automatically discarded; instead, they indicate where either additional experimental excitation, additional measurements, or stronger literature priors are needed before those parameters should be used as flexible MPCC degrees of freedom.


In [ ]:
final = pd.read_csv(RESULTS / 'adequacy_final_selected.csv')
cols = [c for c in ['group','state','species','pool','n','relative_rmse','relative_bias','corr','adequacy_flag','adequacy_reason'] if c in final.columns]
final.sort_values(['adequacy_flag','relative_rmse'], ascending=[False, False])[cols]


## FIM and estimability

The FIM is built from current-data residuals using finite differences in log-parameter space:

$$J_{ij}\approx\frac{r_i(\theta_j e^h)-r_i(\theta_j e^{-h})}{2h}.$$

The eigenspectrum of $F$ diagnoses practical identifiability. The weakest eigenvectors identify confounded parameter combinations. The approximate log-standard deviation is read from a stabilized inverse FIM:

$$\Sigma_{\log\theta}\approx F^{-1}.$$

Parameters are classified as well estimated, moderate, weak but actionable, or weak/confounded using the same thresholds as the previous pilot aroma workflow.


In [ ]:
json.load(open(RESULTS / 'target_parameters_global_selected.json'))

In [ ]:
pd.read_csv(RESULTS / 'parameter_estimability_global_current.csv')

In [ ]:
pd.read_csv(RESULTS / 'weak_directions_global_current.csv')

## Model-based DOE

Candidate natural-must designs are ranked by adding each candidate FIM to the current-data FIM. The hybrid score keeps D-optimality as the main objective while penalizing weak minimum eigen-directions:

$$\Phi_{hybrid}=\log\det(F)-2|\log(\lambda_{min}/\lambda_{max})|-0.05\log(\operatorname{tr}(F^{-1})).$$

The greedy campaign then adds the experiment that maximizes this score at each step. This is homologous to the previous `doe_multiexperiment` logic: current-data FIM acts as the prior information matrix and candidate experiments add information sequentially.


In [ ]:
pd.read_csv(RESULTS / 'candidate_ranking_global_selected.csv').head(15)

In [ ]:
pd.read_csv(RESULTS / 'selected_campaign_hybrid_global_selected.csv')

## Interpretation

The final model combines:

- The primary fermentation and glycerol model used in the previous pilot calibration.
- The selected secondary structure `secondary_full_chem_o2fixed`.
- The selected aroma structure `ea_ethanol_nlimited` with Antoine plus UNIFAC water-ethanol partition and CO2-driven stripping.

The practical decision rule is:

- If a state is structurally adequate and a parameter is weak in the FIM, prioritize DOE excitation.
- If a state remains structurally flagged, do not interpret weak FIM directions as only a sampling/design problem; first improve model structure, measurement interpretation, or priors.
- If a parameter is active at a bound and dominates weak directions, fix or regularize it before using the model as an MPCC constraint.
